# Full Quantum Chemistry Workflow with Maestro

This notebook walks through a **complete, publication-quality quantum chemistry pipeline** using `qoro-maestro-pyscf`. We'll model BeH₂ (beryllium dihydride) from scratch — starting with the molecule definition and ending with molecular properties.

### What You'll Learn

1. How PySCF and Maestro work together
2. What each step in a CASSCF calculation does
3. How NEVPT2 adds dynamic correlation *for free* on top of VQE
4. How to extract molecular properties (dipole moments, natural orbitals) from the quantum simulation

### Prerequisites

```bash
pip install qoro-maestro-pyscf matplotlib
```

---

## Step 0: Imports

We need three things:
- **PySCF** — the classical chemistry engine (molecule, integrals, CASSCF)
- **MaestroSolver** — our quantum VQE solver that drops into PySCF
- **Properties** — to compute dipole moments and natural orbitals from the VQE result

In [2]:
import numpy as np
import time
import matplotlib.pyplot as plt

from pyscf import gto, scf, mcscf, mrpt

from qoro_maestro_pyscf import MaestroSolver
from qoro_maestro_pyscf.properties import (
    compute_dipole_moment,
    compute_natural_orbitals,
)

# Change to "gpu" if you have a Maestro GPU license
BACKEND = "cpu"

# Plot style
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#c9d1d9',
    'text.color': '#c9d1d9',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'grid.color': '#21262d',
    'font.family': 'sans-serif',
    'font.size': 12,
})

---

## Step 1: Define the Molecule

We'll study **BeH₂** — beryllium dihydride in a linear geometry. This molecule is a classic benchmark because:

- At equilibrium, it's well-described by a single Slater determinant
- But the Be 2s and 2p orbitals are nearly degenerate, creating **multi-reference character**
- This near-degeneracy is exactly what CASSCF + VQE is designed to handle

We use the minimal STO-3G basis set to keep the quantum circuit small enough to simulate.

In [3]:
mol = gto.M(
    atom="""
        Be  0.000  0.000  0.000
        H   0.000  0.000  1.334
        H   0.000  0.000 -1.334
    """,
    basis="sto-3g",
    symmetry=False,
    verbose=0,
)

print(f"Atoms          : {mol.natm}")
print(f"Electrons      : {mol.nelectron}")
print(f"Basis functions: {mol.nao_nr()}")
print(f"Spin orbitals  : {2 * mol.nao_nr()}")

Atoms          : 3
Electrons      : 6
Basis functions: 7
Spin orbitals  : 14


---

## Step 2: Hartree-Fock

Hartree-Fock (HF) gives us the **mean-field** approximation — it treats each electron as moving in the average field of all the others. This is fast but misses **electron correlation** (the fact that electrons avoid each other).

The HF result gives us:
- A reference energy (baseline to improve upon)
- Molecular orbitals (starting point for CASSCF)

In [4]:
hf = scf.RHF(mol).run()

print(f"HF energy      : {hf.e_tot:+.10f} Ha")
print(f"HF converged   : {hf.converged}")

HF energy      : -15.5598384372 Ha
HF converged   : True


---

## Step 3: CASSCF with Maestro VQE

This is **the quantum step**.

### What is CASSCF?

CASSCF = **Complete Active Space Self-Consistent Field**. It works in two nested loops:

1. **Inner loop (CI)**: Solve the Schrödinger equation exactly within a small "active space" of orbitals. This is where our VQE runs on Maestro.
2. **Outer loop (SCF)**: Rotate the molecular orbitals to minimise the total energy. This uses the **density matrices** (RDMs) from the VQE.

### Active Space: (2e, 3o)

We put **2 electrons** into **3 orbitals** — the Be 2s, one 2p orbital, and the antibonding combination with H 1s. This gives us **6 qubits** (2 spin-orbitals per spatial orbital).

```
     ┌───────────────────────────┐
     │     Active Space          │
     │                           │
  ↑↓ │  ─── σ*        (virtual) │
     │  ─── 2p         (virtual) │
  ↑↓ │  ─── 2s/σ    (occupied)  │
     │                           │
     └───────────────────────────┘
  ↑↓    ─── 1s         (frozen core)
```

The `MaestroSolver` handles everything:
- Maps the integrals to a qubit Hamiltonian (Jordan-Wigner)
- Builds the parameterised quantum circuit (ansatz)
- Runs VQE optimisation on Maestro's backend
- Returns density matrices for the CASSCF orbital update

In [5]:
norb = 3   # active spatial orbitals
nelec = 2  # active electrons

print(f"Active space   : ({nelec}e, {norb}o) → {2 * norb} qubits")

cas = mcscf.CASSCF(hf, norb, nelec)
cas.fcisolver = MaestroSolver(
    ansatz="hardware_efficient",
    ansatz_layers=2,
    backend=BACKEND,
    maxiter=200,
    verbose=True,
)
cas.max_cycle_macro = 15  # limit CASSCF macro-iterations
cas.verbose = 0

t0 = time.perf_counter()
casscf_e = cas.kernel()[0]
casscf_time = time.perf_counter() - t0

print(f"\nCASSCF energy   : {casscf_e:+.10f} Ha")
print(f"Time            : {casscf_time:.1f}s")

Active space   : (2e, 3o) → 6 qubits
  [MaestroSolver] Backend : CPU/QCSim (statevector)
  [MaestroSolver] Qubits  : 6
  [MaestroSolver] Ansatz  : hardware_efficient
  [MaestroSolver] Params  : 24
  [MaestroSolver] Paulis  : 39
    iter    1  |  E = -2.2089725410
    iter   20  |  E = -2.1868589522
    iter   40  |  E = -2.2057006149
    iter   60  |  E = -2.2102017846
    iter   80  |  E = -2.2120054833
    iter  100  |  E = -2.2135320158
    iter  120  |  E = -2.2129166627
    iter  140  |  E = -2.2136617076
    iter  160  |  E = -2.2136902075
    iter  180  |  E = -2.2137097694
    iter  200  |  E = -2.2137403049
  [MaestroSolver] Converged : False
  [MaestroSolver] E(VQE)    : -2.2137403049
  [MaestroSolver] E(total)  : -16.4973626261
  [MaestroSolver] Time      : 0.15s
  [MaestroSolver] Backend : CPU/QCSim (statevector)
  [MaestroSolver] Qubits  : 6
  [MaestroSolver] Ansatz  : hardware_efficient
  [MaestroSolver] Params  : 24
  [MaestroSolver] Paulis  : 47
    iter    1  |  E = -2

---

## Step 4: NEVPT2 — Adding Dynamic Correlation

CASSCF captures **static correlation** (the multi-reference character from near-degenerate orbitals). But it misses **dynamic correlation** — the short-range electron-electron repulsion that comes from the remaining orbitals outside the active space.

**NEVPT2** (N-Electron Valence Perturbation Theory, 2nd order) adds this dynamic part using perturbation theory. It only needs the RDMs from CASSCF — which our VQE already computed.

This step is **purely classical**. The quantum circuit is only involved in Step 3.

In [6]:
t0 = time.perf_counter()
nevpt2_corr = mrpt.NEVPT(cas).kernel()
nevpt2_time = time.perf_counter() - t0

nevpt2_e = casscf_e + nevpt2_corr

print(f"NEVPT2 correction : {nevpt2_corr:+.10f} Ha")
print(f"Total energy      : {nevpt2_e:+.10f} Ha")
print(f"Time              : {nevpt2_time:.1f}s")

  [MaestroSolver] Backend : CPU/QCSim (statevector)
  [MaestroSolver] Qubits  : 6
  [MaestroSolver] Ansatz  : hardware_efficient


IndexError: too many indices for array: array is 2-dimensional, but 4 were indexed

---

## Step 5: Molecular Properties

A bare energy number is rarely enough for a chemist. We need **observables** that connect to experiment.

### Dipole Moment

The dipole moment $\boldsymbol{\mu}$ tells you how charge is distributed in the molecule. It's computed from the 1-RDM:

$$\mu_x = \sum_{pq} D_{pq} \langle p | x | q \rangle + \text{nuclear contribution}$$

For BeH₂ in a linear symmetric geometry, the dipole should be **zero by symmetry**.

### Natural Orbital Occupations

Natural orbitals are the eigenvectors of the 1-RDM. Their eigenvalues (occupations) tell you about the correlation in the system:

- **Occupation ≈ 2.0**: doubly occupied, well-described by HF
- **Occupation ≈ 0.0**: empty, not contributing
- **Occupation between 0.3 and 1.7**: **fractional** — this orbital is correlated, and you need multi-reference methods

In [ ]:
# Get the 1-RDM from the VQE
ci = cas.ci
rdm1 = cas.fcisolver.make_rdm1(ci, norb, (1, 1))

# Dipole moment
dipole, mag = compute_dipole_moment(mol, cas.mo_coeff, rdm1)
print(f"Dipole moment")
print(f"  x: {dipole[0]:+.6f} D")
print(f"  y: {dipole[1]:+.6f} D")
print(f"  z: {dipole[2]:+.6f} D")
print(f"  |μ| = {mag:.6f} D")
print(f"  (Should be ~0 by symmetry)")

# Natural orbital occupations
occ, _ = compute_natural_orbitals(rdm1)
print(f"\nNatural orbital occupations:")
for i, n in enumerate(occ):
    bar = '█' * int(n * 25)
    label = 'correlated' if 0.3 < n < 1.7 else 'occupied' if n > 1.5 else 'empty'
    print(f"  NO {i}: {n:.4f}  {bar}  ({label})")

---

## Summary: Energy Hierarchy

Let's compare all the methods. Each layer of theory adds more accuracy:

In [ ]:
# Classical FCI reference (exact within this active space)
cas_fci = mcscf.CASCI(hf, norb, nelec)
cas_fci.verbose = 0
fci_e = cas_fci.kernel()[0]

print(f"{'Method':<22s}  {'Energy (Ha)':>14s}  {'Error to FCI':>12s}")
print(f"{'─' * 52}")
for label, e in [
    ('HF', hf.e_tot),
    ('CASSCF (Maestro)', casscf_e),
    ('CASSCF + NEVPT2', nevpt2_e),
    ('FCI (exact)', fci_e),
]:
    err = abs(e - fci_e) * 1000
    marker = ' ◀ GPU' if 'Maestro' in label else ''
    print(f"{label:<22s}  {e:+14.8f}  {err:10.2f} mHa{marker}")

---

## Visualisation: Energy Ladder

How does each method close the gap to exact (FCI) energy?

In [ ]:
methods = ['HF', 'CASSCF\n(Maestro VQE)', 'CASSCF +\nNEVPT2', 'FCI\n(exact)']
energies = [hf.e_tot, casscf_e, nevpt2_e, fci_e]
errors = [abs(e - fci_e) * 1000 for e in energies]

# Colours: gradient from red (bad) to green (exact)
colors = ['#f85149', '#d29922', '#3fb950', '#58a6ff']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={'width_ratios': [3, 2]})

# --- Left: Energy ladder ---
bars = ax1.barh(methods, errors, color=colors, edgecolor='#30363d', height=0.6)
ax1.set_xlabel('Error to FCI (mHa)', fontsize=13)
ax1.set_title('Energy Error by Method', fontsize=15, fontweight='bold', pad=15)
ax1.invert_yaxis()
ax1.set_xlim(0, max(errors) * 1.15)

# Add value labels
for bar, err in zip(bars, errors):
    if err > 0.01:
        ax1.text(bar.get_width() + max(errors) * 0.02, bar.get_y() + bar.get_height() / 2,
                 f'{err:.1f} mHa', va='center', fontsize=11, color='#c9d1d9')
    else:
        ax1.text(max(errors) * 0.02, bar.get_y() + bar.get_height() / 2,
                 'exact', va='center', fontsize=11, color='#58a6ff', fontweight='bold')

# Chemical accuracy line
ax1.axvline(x=1.6, color='#f0883e', linestyle='--', linewidth=1.5, alpha=0.7)
ax1.text(1.6, -0.45, ' chemical\n accuracy', fontsize=9, color='#f0883e', va='top')

# --- Right: Absolute energies ---
y_pos = range(len(methods))
ax2.scatter(energies, y_pos, c=colors, s=120, zorder=5, edgecolors='#c9d1d9', linewidth=1)
ax2.set_xlabel('Total Energy (Ha)', fontsize=13)
ax2.set_title('Absolute Energies', fontsize=15, fontweight='bold', pad=15)
ax2.set_yticks(list(y_pos))
ax2.set_yticklabels(methods)
ax2.invert_yaxis()
ax2.grid(axis='x', alpha=0.3)

# FCI reference line
ax2.axvline(x=fci_e, color='#58a6ff', linestyle=':', linewidth=1, alpha=0.5)

for i, e in enumerate(energies):
    ax2.text(e, i - 0.3, f'{e:.4f}', fontsize=9, ha='center', color='#8b949e')

plt.tight_layout()
plt.savefig('energy_ladder.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved: energy_ladder.png')

---

## Visualisation: VQE Convergence

How did the VQE optimiser converge? The `energy_history` attribute tracks the energy at each iteration.

In [ ]:
history = cas.fcisolver.energy_history

fig, ax = plt.subplots(figsize=(10, 5))

iters = np.arange(1, len(history) + 1)

# Main convergence curve
ax.plot(iters, history, color='#58a6ff', linewidth=2, label='VQE energy')

# Fill below the curve
ax.fill_between(iters, history, min(history) - 0.01 * abs(min(history)),
                alpha=0.1, color='#58a6ff')

# FCI reference line
ax.axhline(y=fci_e, color='#3fb950', linestyle='--', linewidth=1.5,
           label=f'FCI = {fci_e:.6f} Ha', alpha=0.8)

# Mark the final energy
final_e = history[-1]
ax.scatter([len(history)], [final_e], color='#f0883e', s=80, zorder=5,
           edgecolors='#c9d1d9', linewidth=1)
ax.annotate(f'Final: {final_e:.6f} Ha',
            xy=(len(history), final_e),
            xytext=(len(history) * 0.7, final_e + (history[0] - final_e) * 0.15),
            fontsize=10, color='#f0883e',
            arrowprops=dict(arrowstyle='->', color='#f0883e', lw=1.5))

ax.set_xlabel('VQE Iteration', fontsize=13)
ax.set_ylabel('Energy (Ha)', fontsize=13)
ax.set_title('VQE Convergence on Maestro', fontsize=15, fontweight='bold', pad=15)
ax.legend(loc='upper right', fontsize=11, framealpha=0.8,
          edgecolor='#30363d', facecolor='#161b22')
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('vqe_convergence.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f'Converged in {len(history)} iterations')
print(f'Saved: vqe_convergence.png')

---

## Visualisation: Natural Orbital Occupations

Natural orbital occupations reveal the multi-reference character of the wavefunction. Occupations near 2.0 or 0.0 are "boring" (HF gets them right). Fractional occupations flag the correlated orbitals that justify using a quantum solver.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

x = np.arange(len(occ))
labels = [f'NO {i}' for i in range(len(occ))]

# Colour by correlation character
bar_colors = []
for n in occ:
    if n > 1.7:
        bar_colors.append('#3fb950')   # occupied → green
    elif n < 0.3:
        bar_colors.append('#8b949e')   # empty → grey
    else:
        bar_colors.append('#f85149')   # correlated → red

bars = ax.bar(x, occ, color=bar_colors, edgecolor='#30363d', width=0.6)

# Reference lines
ax.axhline(y=2.0, color='#3fb950', linestyle=':', linewidth=1, alpha=0.4)
ax.axhline(y=0.0, color='#8b949e', linestyle=':', linewidth=1, alpha=0.4)

# Correlated bands
ax.axhspan(0.3, 1.7, alpha=0.05, color='#f85149')
ax.text(len(occ) - 0.5, 1.0, 'correlated\nregion', fontsize=9,
        color='#f85149', ha='center', va='center', alpha=0.7, fontstyle='italic')

# Value labels on bars
for bar, n in zip(bars, occ):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
            f'{n:.3f}', ha='center', fontsize=11, color='#c9d1d9', fontweight='bold')

ax.set_xlabel('Natural Orbital', fontsize=13)
ax.set_ylabel('Occupation Number', fontsize=13)
ax.set_title('Natural Orbital Occupations (VQE)', fontsize=15, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(-0.1, 2.3)
ax.grid(axis='y', alpha=0.2)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#3fb950', edgecolor='#30363d', label='Occupied (≈2.0)'),
    Patch(facecolor='#f85149', edgecolor='#30363d', label='Correlated (0.3–1.7)'),
    Patch(facecolor='#8b949e', edgecolor='#30363d', label='Empty (≈0.0)'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=10,
          framealpha=0.8, edgecolor='#30363d', facecolor='#161b22')

plt.tight_layout()
plt.savefig('natural_orbitals.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved: natural_orbitals.png')

---

## What Just Happened?

Let's recap the full pipeline:

| Step | Method | Runs on | What it captures |
|------|--------|---------|------------------|
| 1 | Molecule definition | — | Nuclear geometry |
| 2 | Hartree-Fock | CPU | Mean-field energy, starting orbitals |
| 3 | **CASSCF + VQE** | **Maestro GPU** | **Static correlation** (multi-reference character) |
| 4 | NEVPT2 | CPU | Dynamic correlation (electron cusp) |
| 5 | Properties | CPU | Dipole moment, natural orbital occupations |

The quantum circuit is only involved in **Step 3** — everything else is classical. But that one step is critical: it captures the strongly-correlated physics that breaks mean-field methods.

### Why Maestro?

- **GPU acceleration**: VQE evaluations run on NVIDIA GPUs via Maestro
- **MPS mode**: For larger active spaces, use `simulation="mps"` to go beyond 30 qubits
- **Drop-in replacement**: The VQE solver plugs directly into PySCF's standard CASSCF workflow
- **No Qiskit needed**: Pure PySCF + Maestro stack

---

## Next Steps

Try modifying this notebook:

1. **Change the molecule**: Replace BeH₂ with H₂O, LiH, or N₂
2. **Increase the active space**: Use `norb=4, nelec=4` for 8 qubits
3. **Try UCCSD ansatz**: Change to `ansatz="uccsd"` for chemistry-motivated circuits
4. **Enable GPU**: Set `BACKEND = "gpu"` (requires a Maestro license key)
5. **Use MPS**: Add `simulation="mps", mps_bond_dim=64` for larger systems

```python
# Example: H₂O with UCCSD on GPU MPS
cas.fcisolver = MaestroSolver(
    ansatz="uccsd",
    backend="gpu",
    simulation="mps",
    mps_bond_dim=128,
)
```